# CAP182 SS2 Common Issues & Key Concepts

This notebook addresses four recurring issues from the SS2 assessment.  
We use the **Bank Marketing** dataset from the UCI Machine Learning Repository throughout.

**Contents**
1. Data Leakage — the `duration` trap
2. What `class_weight` actually does
3. Confidence Intervals for a single model
4. Statistical comparison between models

## Setup

In [1]:
# Install dependencies (run once)
!pip install ucimlrepo xgboost -q

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RepeatedStratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print('All imports successful ✓')

All imports successful ✓


## Load the Bank Marketing dataset

In [3]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=222)
df = dataset.data.original.copy()

print(f'Shape: {df.shape}')
df.head()

ConnectionError: Error connecting to server

In [ ]:
# Encode the target: 'yes' → 1, 'no' → 0
df['y'] = df['y'].map({'yes': 1, 'no': 0})

print('Target distribution:')
print(df['y'].value_counts())
print(f'\nClass balance: {df["y"].mean():.1%} positive (subscribed)')

Target distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Class balance: 11.7% positive (subscribed)


---
# 1. Data Leakage — the `duration` trap

### What is data leakage?

**Data leakage** occurs when your model has access to information during training that it would **not** have at prediction time.

This doesn't just mean accidentally including the target variable. A feature can cause leakage even though it is *not* the target,
if that feature is only **knowable after the outcome** has already been determined.

### The `duration` column

In the Bank Marketing dataset, `duration` is the **duration of the last phone call in seconds**.

Why is this leakage?

- You only know `duration` **after** the call has ended.
- If `duration = 0`, no call was made, so the answer is **always** 'no'.
- A longer call usually means the client was interested → more likely 'yes'.

The [dataset documentation](https://archive.ics.uci.edu/dataset/222/bank+marketing) explicitly warns:

> *"this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed."*

In a real deployment, you would be trying to predict **before** dialling — so `duration` would not exist yet.

In [ ]:
# Look at the relationship between duration and the target
print('Average call duration by outcome:\n')
print(df.groupby('y')['duration'].agg(['mean', 'median', 'std']).round(1))

Average call duration by outcome:

    mean  median    std
y                      
0  221.2   164.0  207.4
1  537.3   426.0  392.5


### Impact: model performance WITH vs WITHOUT `duration`

Let's train the same model with and without `duration` to see how inflated the scores become.

In [ ]:
# Quick one-hot encoding for all categorical columns
def prepare_features(dataframe, drop_duration=False):
    """One-hot encode categoricals and return X, y."""
    df_temp = dataframe.copy()
    y = df_temp.pop('y')
    if drop_duration:
        df_temp = df_temp.drop(columns=['duration'])
    X = pd.get_dummies(df_temp, drop_first=True)
    return X, y

# --- WITH duration (leakage!) ---
X_leak, y = prepare_features(df, drop_duration=False)
X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
rf_leak = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_leak.fit(X_train, y_train)
y_pred_leak = rf_leak.predict(X_test)

print('=== WITH duration (leaky) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_leak):.4f}')
print(f'F1 (pos) : {f1_score(y_test, y_pred_leak):.4f}')

=== WITH duration (leaky) ===
Accuracy : 0.9053
F1 (pos) : 0.4843


In [ ]:
# --- WITHOUT duration (correct) ---
X_clean, y = prepare_features(df, drop_duration=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
rf_clean = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_clean.fit(X_train, y_train)
y_pred_clean = rf_clean.predict(X_test)

print('=== WITHOUT duration (correct) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_clean):.4f}')
print(f'F1 (pos) : {f1_score(y_test, y_pred_clean):.4f}')

=== WITHOUT duration (correct) ===
Accuracy : 0.8946
F1 (pos) : 0.3263


### Takeaway

Including `duration` artificially inflates your metrics.  
Your model looks great on paper but **would fail completely in production**,
because you cannot know `duration` before making the call.

**Rule of thumb:** For every feature, ask yourself:  
*'Would I have this value available at the moment I need to make a prediction?'*  
If not → **drop it**.

---
# 2. What does `class_weight` actually do?

Many of you used `class_weight='balanced'` in your models but then stated  
*'I did not use any imbalance handling.'*

**`class_weight='balanced'` IS an imbalance handling technique.** Here is what it does.

### The idea

In an imbalanced dataset the model sees far more examples of the majority class,
so it tends to predict that class more often.  
`class_weight='balanced'` tells the model to **penalise mistakes on the minority class more heavily**,
so that each class contributes equally to the loss function.

### The formula

When you set `class_weight='balanced'`, scikit-learn computes a weight for each class:

$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_{\text{samples in class } c}}$$

- Majority class → weight **< 1** (penalised less per sample)  
- Minority class → weight **> 1** (penalised more per sample)

In [ ]:
# Show the class distribution
n_samples = len(y_train)
n_classes = 2
counts = y_train.value_counts().sort_index()

print(f'Training samples : {n_samples}')
print(f'Class 0 (no)     : {counts[0]}')
print(f'Class 1 (yes)    : {counts[1]}')
print(f'Imbalance ratio  : {counts[0] / counts[1]:.1f} : 1')

Training samples : 31647
Class 0 (no)     : 27945
Class 1 (yes)    : 3702
Imbalance ratio  : 7.5 : 1


In [ ]:
# Manually compute what class_weight='balanced' calculates
w0 = n_samples / (n_classes * counts[0])
w1 = n_samples / (n_classes * counts[1])

print('Manually computed balanced weights:')
print(f'  Weight for class 0 (no)  = {w0:.4f}')
print(f'  Weight for class 1 (yes) = {w1:.4f}')

# Verify with sklearn
from sklearn.utils.class_weight import compute_class_weight
sk_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
print(f'\nsklearn balanced weights : {sk_weights}')

Manually computed balanced weights:
  Weight for class 0 (no)  = 0.5662
  Weight for class 1 (yes) = 4.2743

sklearn balanced weights : [0.56623725 4.27431118]


### Impact on predictions

Let's compare a Random Forest **without** vs **with** `class_weight='balanced'`.

In [ ]:
# WITHOUT class_weight
rf_no_weight = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE
)
rf_no_weight.fit(X_train, y_train)

print('=== WITHOUT class_weight ===')
print(classification_report(y_test, rf_no_weight.predict(X_test),
                            target_names=['no', 'yes']))

=== WITHOUT class_weight ===
              precision    recall  f1-score   support

          no       0.90      0.98      0.94     11977
         yes       0.65      0.22      0.33      1587

    accuracy                           0.89     13564
   macro avg       0.78      0.60      0.63     13564
weighted avg       0.87      0.89      0.87     13564



In [ ]:
# WITH class_weight='balanced'
rf_weighted = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE,
    class_weight='balanced'
)
rf_weighted.fit(X_train, y_train)

print('=== WITH class_weight="balanced" ===')
print(classification_report(y_test, rf_weighted.predict(X_test),
                            target_names=['no', 'yes']))

=== WITH class_weight="balanced" ===
              precision    recall  f1-score   support

          no       0.90      0.99      0.94     11977
         yes       0.67      0.19      0.29      1587

    accuracy                           0.89     13564
   macro avg       0.79      0.59      0.62     13564
weighted avg       0.88      0.89      0.87     13564



### Takeaway

| Technique | What it is |
|---|---|
| `class_weight='balanced'` | Adjusts the loss function — **this IS imbalance handling** |
| SMOTE / oversampling | Creates synthetic samples for the minority class |
| Undersampling | Removes samples from the majority class |

All three are valid approaches to imbalanced data. If you used `class_weight='balanced'`,
you **did** handle imbalance — own it and explain it!

---
# 3. Confidence Intervals

A single accuracy or F1 score is a **point estimate** — it tells you nothing about
how much that number might vary.  
**Confidence intervals** give a range that likely contains the true performance.

### Approach: Repeated Stratified K-Fold CV

1. Run stratified k-fold cross-validation **multiple times** (different random splits).  
2. Collect all the fold scores.  
3. Compute the mean and the 95% CI using the *t*-distribution.

In [ ]:
from xgboost import XGBClassifier

def confidence_interval(model, X, y, scoring='f1', n_splits=5, n_repeats=10):
    """
    Compute a 95% confidence interval for `scoring`
    using Repeated Stratified K-Fold CV.
    """
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE
    )
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)

    mean  = scores.mean()
    se    = scores.std() / np.sqrt(len(scores))
    ci_lo, ci_hi = stats.t.interval(
        confidence=0.95, df=len(scores) - 1, loc=mean, scale=se
    )
    return mean, ci_lo, ci_hi, scores

print('confidence_interval() defined ✓')

confidence_interval() defined ✓


### Logistic Regression

The logistic regression achieved a mean F1 of 0.27. The 95% confidence interval of [0.2662, 0.2744] tells us that we can be 95% confident the model’s true F1 performance falls within this narrow range, suggesting a stable estimate.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

mean_lr, lo_lr, hi_lr, scores_lr = confidence_interval(lr, X_clean, y)

print(f'Logistic Regression  F1 = {mean_lr:.4f}  95% CI [{lo_lr:.4f}, {hi_lr:.4f}]')

Logistic Regression  F1 = 0.2703  95% CI [0.2662, 0.2744]


### Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=6, random_state=RANDOM_STATE)

mean_rf, lo_rf, hi_rf, scores_rf = confidence_interval(rf, X_clean, y)

print(f'Random Forest        F1 = {mean_rf:.4f}  95% CI [{lo_rf:.4f}, {hi_rf:.4f}]')

Random Forest        F1 = 0.2697  95% CI [0.2648, 0.2747]


### XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=100, eval_metric='logloss',
    use_label_encoder=False, random_state=RANDOM_STATE, verbosity=0
)

mean_xgb, lo_xgb, hi_xgb, scores_xgb = confidence_interval(xgb, X_clean, y)

print(f'XGBoost              F1 = {mean_xgb:.4f}  95% CI [{lo_xgb:.4f}, {hi_xgb:.4f}]')

XGBoost              F1 = 0.3585  95% CI [0.3543, 0.3627]


### Summary table

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Mean F1': [mean_lr, mean_rf, mean_xgb],
    '95% CI Lower': [lo_lr, lo_rf, lo_xgb],
    '95% CI Upper': [hi_lr, hi_rf, hi_xgb],
    '95% CI Lower Diff': [mean_lr - lo_lr, mean_rf - lo_rf, mean_xgb - lo_xgb],
    '95% CI Upper Diff': [hi_lr - mean_lr, hi_rf - mean_rf, hi_xgb - mean_xgb],
	'95% CI Diff': [hi_lr - lo_lr, hi_rf - lo_rf, hi_xgb - lo_xgb],
})
results['Mean F1'] = results['Mean F1'].round(4)
results['95% CI Lower'] = results['95% CI Lower'].round(4)
results['95% CI Upper'] = results['95% CI Upper'].round(4)

print(results.to_string(index=False))

              Model  Mean F1  95% CI Lower  95% CI Upper  95% CI Lower Diff  95% CI Upper Diff  95% CI Diff
Logistic Regression   0.2703        0.2662        0.2744           0.004088           0.004088     0.008175
      Random Forest   0.2697        0.2648        0.2747           0.004927           0.004927     0.009855
            XGBoost   0.3585        0.3543        0.3627           0.004238           0.004238     0.008476


### Takeaway

Always report confidence intervals alongside point estimates.  
A model with F1 = 0.42 ± 0.01 tells a very different story  
than F1 = 0.42 ± 0.10.

---
# 4. Statistical Comparison Between Models

If Model A has F1 = 0.43 and Model B has F1 = 0.41, is A *really* better?  
You need a **statistical test** to answer that.

### Strategy

1. Evaluate both models on the **same** CV folds (paired observations).  
2. **Paired *t*-test** — simple, assumes normally distributed score differences.  
3. **Mann–Whitney U test** — non-parametric, no normality assumption.

We'll compare all three models from Section 3.

### Step 1 — Collect paired CV scores

We must use the **same folds** for each model so the comparisons are paired.

In [ ]:
# Use the same CV folds for all models
cv = RepeatedStratifiedKFold(
    n_splits=5, n_repeats=10, random_state=RANDOM_STATE
)

scores_lr  = cross_val_score(lr,  X_clean, y, cv=cv, scoring='f1')
scores_rf  = cross_val_score(rf,  X_clean, y, cv=cv, scoring='f1')
scores_xgb = cross_val_score(xgb, X_clean, y, cv=cv, scoring='f1')

print(f'LR  scores: mean={scores_lr.mean():.4f}  std={scores_lr.std():.4f}')
print(f'RF  scores: mean={scores_rf.mean():.4f}  std={scores_rf.std():.4f}')
print(f'XGB scores: mean={scores_xgb.mean():.4f}  std={scores_xgb.std():.4f}')

LR  scores: mean=0.2703  std=0.0144
RF  scores: mean=0.2697  std=0.0173
XGB scores: mean=0.3585  std=0.0149


### Step 2 — Paired *t*-test

The **paired *t*-test** checks whether the mean difference in scores between two models
is significantly different from zero.

- **H₀**: The two models perform equally (mean difference = 0).  
- **H₁**: The two models perform differently (mean difference ≠ 0).  
- Reject H₀ if *p* < 0.05.

In [ ]:
def paired_ttest(scores_a, scores_b, name_a, name_b):
    """Run a paired t-test and print the result."""
    t_stat, p_value = stats.ttest_rel(scores_a, scores_b, alternative='less')
    sig = '✓ Significant' if p_value < 0.05 else '✗ Not significant'
    print(f'{name_a} < {name_b}:')
    print(f'  t-statistic = {t_stat:.4f}')
    print(f'  p-value     = {p_value:.4f}  ({sig} at α=0.05)')
    print()

paired_ttest(scores_lr,  scores_rf,  'LR',  'RF')
paired_ttest(scores_lr,  scores_xgb, 'LR',  'XGB')
paired_ttest(scores_rf,  scores_xgb, 'RF',  'XGB')

LR < RF:
  t-statistic = 0.2076
  p-value     = 0.5818  (✗ Not significant at α=0.05)

LR < XGB:
  t-statistic = -43.6813
  p-value     = 0.0000  (✓ Significant at α=0.05)

RF < XGB:
  t-statistic = -37.6758
  p-value     = 0.0000  (✓ Significant at α=0.05)



### Step 3 — Mann–Whitney U test

The **Mann–Whitney U test** is a non-parametric alternative.  
It does not assume the score differences are normally distributed,
making it more robust for small sample sizes.

- **H₀**: The distributions of scores for both models are equal.  
- **H₁**: The distributions differ.

In [ ]:
def mann_whitney(scores_a, scores_b, name_a, name_b):
    """Run a Mann-Whitney U test and print the result."""
    u_stat, p_value = stats.mannwhitneyu(
        scores_a, scores_b, alternative='less'
    )
    sig = '✓ Significant' if p_value < 0.05 else '✗ Not significant'
    print(f'{name_a} < {name_b}:')
    print(f'  U-statistic = {u_stat:.1f}')
    print(f'  p-value     = {p_value:.4f}  ({sig} at α=0.05)')
    print()

mann_whitney(scores_lr,  scores_rf,  'LR',  'RF')
mann_whitney(scores_lr,  scores_xgb, 'LR',  'XGB')
mann_whitney(scores_rf,  scores_xgb, 'RF',  'XGB')

LR < RF:
  U-statistic = 1250.0
  p-value     = 0.5014  (✗ Not significant at α=0.05)

LR < XGB:
  U-statistic = 0.0
  p-value     = 0.0000  (✓ Significant at α=0.05)

RF < XGB:
  U-statistic = 0.0
  p-value     = 0.0000  (✓ Significant at α=0.05)



### When to use which test?

| Test | Assumptions | Best for |
|---|---|---|
| **Paired *t*-test** | Score differences are approximately normal | Large *k* × repeats, quick check |
| **Mann–Whitney U** | No normality assumption (non-parametric) | Small samples, skewed distributions |

Both are valid. If in doubt, report both — if they agree, you can be more confident.

---
# Summary

| # | Concept | Key point |
|---|---|---|
| 1 | **Data leakage** | Drop any feature you wouldn't have at prediction time — `duration` is a classic example |
| 2 | **`class_weight`** | `class_weight='balanced'` **is** an imbalance-handling technique — acknowledge it |
| 3 | **Confidence intervals** | Always report CIs alongside point estimates using repeated CV |
| 4 | **Statistical comparison** | Use paired *t*-test or Mann–Whitney U to claim one model outperforms another |